# Analysis of the missing data of the dataset

In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv("3SudokuCombined_cleaned.csv")

In [3]:
before_table = pd.crosstab(df["Before"], 'count', dropna = False)
before_table["proportion"] = before_table["count"]/len(df) 
before_table

col_0,count,proportion
Before,,
No,238,0.140165
Out_3,515,0.303298
Within_3,348,0.204947
NaN,597,0.351590


35.15% of the Before variable is missing 

In [4]:
logic_table = pd.crosstab(df["Logic"], "count", dropna = False)
logic_table["proportions"] = logic_table["count"]/len(df)
logic_table

col_0,count,proportions
Logic,,
Indifferent,308,0.181390
No,111,0.065371
Yes,682,0.401649
NaN,597,0.351590


Again, 35.16% of the Logic variable is missing. Let's look at what's happening, since this seems like an MNAR type of missigness.

In [5]:
#we tale all the rows where logic and before have missing values
df["Missing"] = (df["Logic"].isna()) & (df["Before"].isna()) 
df

,Before,Type,Logic,Class,Time,Correct,Missing
0,NaN,Numbers,NaN,1,NaN,No,True
1,NaN,Greek,NaN,1,NaN,No,True
2,NaN,Greek,NaN,1,252.0,No,True
3,NaN,Letters,NaN,1,NaN,No,True
4,NaN,Letters,NaN,1,88.0,No,True
...,...,...,...,...,...,...,...
1693,Within_3,Letters,Indifferent,8,210.0,Yes,False
1694,Out_3,Letters,Yes,8,206.0,No,False
1695,Out_3,Numbers,Yes,8,163.0,Yes,False
1696,Out_3,Numbers,Yes,8,140.0,Yes,False


In [6]:
result = df.groupby('Logic')['Missing'].mean()
result

Logic
Indifferent    0.0
No             0.0
Yes            0.0
Name: Missing, dtype: float64

In [7]:
result = df.groupby('Class')['Missing'].mean()
result

Class
1    1.000000
2    1.000000
3    1.000000
4    0.014388
5    0.000000
6    0.000000
7    0.015385
8    0.009434
Name: Missing, dtype: float64

In [8]:
result = df.groupby('Time')['Missing'].mean()
result

Time
31.0     0.0
33.0     0.0
34.0     0.0
35.0     0.0
39.0     0.5
        ... 
412.0    0.0
414.0    1.0
415.0    0.0
417.0    0.0
418.0    0.0
Name: Missing, Length: 347, dtype: float64

In [9]:
result = df.groupby('Correct')['Missing'].mean()
result

Correct
No     0.361516
Yes    0.349077
Name: Missing, dtype: float64

The missingness in `Before` and `Logic` is HIGHLY concentrated in Classes 1, 2  and 3, while Classes 4–8 have little to no missing values (0 - 1.5%). There seems to be a
difference in data collection across class groups, where Classes 1–3 were not  given the `Before` and `Logic` questions. Since the missingness is because of an observed variable (Class), this is Missing Not At Random (MNAR). No statiscal test would be needed. Now, let's proceed with analyzing the missigness of `Time`. 

In [10]:
time_table = pd.crosstab(df["Time"], 'count', dropna = False)
time_table["proportion"] = time_table["count"]/len(df) 
time_table

col_0,count,proportion
Time,,
31.0,1,0.000589
33.0,1,0.000589
34.0,1,0.000589
35.0,2,0.001178
39.0,4,0.002356
...,...,...
414.0,1,0.000589
415.0,1,0.000589
417.0,1,0.000589


In [11]:
df["Missing"] = df["Time"].isna()
df

,Before,Type,Logic,Class,Time,Correct,Missing
0,NaN,Numbers,NaN,1,NaN,No,True
1,NaN,Greek,NaN,1,NaN,No,True
2,NaN,Greek,NaN,1,252.0,No,False
3,NaN,Letters,NaN,1,NaN,No,True
4,NaN,Letters,NaN,1,88.0,No,False
...,...,...,...,...,...,...,...
1693,Within_3,Letters,Indifferent,8,210.0,Yes,False
1694,Out_3,Letters,Yes,8,206.0,No,False
1695,Out_3,Numbers,Yes,8,163.0,Yes,False
1696,Out_3,Numbers,Yes,8,140.0,Yes,False


In [12]:
result = df.groupby('Class')['Missing'].mean()
result

Class
1    0.130178
2    0.225681
3    0.209877
4    0.104317
5    0.092593
6    0.057971
7    0.161538
8    0.044025
Name: Missing, dtype: float64

In [13]:
result = df.groupby('Type')['Missing'].mean()
result

Type
Greek      0.200969
Letters    0.068337
Numbers    0.080201
Symbols    0.131991
Name: Missing, dtype: float64

In [14]:
result = df.groupby('Logic')['Missing'].mean()
result

Logic
Indifferent    0.126623
No             0.234234
Yes            0.030792
Name: Missing, dtype: float64

In [15]:
result = df.groupby('Before')['Missing'].mean()
result

Before
No          0.239496
Out_3       0.046602
Within_3    0.014368
Name: Missing, dtype: float64

In [16]:
result = df.groupby('Correct')['Missing'].mean()
result

Correct
No     0.428571
Yes    0.042066
Name: Missing, dtype: float64

Let's analyze the results: 
1. Class — The probability of `Time` being missing varies  across classes, ranging from ~4% in Class 8 up to ~23% in Class 2. 
2. Type — Greek symbol puzzles are missing Time at ~20%, roughly 3× more than Letters or Numbers. 
3. Logic — Participants who answered "No" to Logic are missing Time at ~23%, while those who answered "Yes" are only missing it at ~3%. 
4. Before - Students who didn't try sudoku before present missing time values around ~24%, whereas students who did play it before (be it outside of the three months period or inside of it) present missing time values around ~1.4-4.7%.
5. Correct - There seems to be strong evidence that when a student does not finish the sudoku the value of time is missing (it is around 43% of the times), whereas it is only 3% when the student does complete the sudoku.

Since the porbability of missing changes depending on the value of other variables such as `Class`, `Logic`, `Before` or `Type`, this is could be consistent with Missing at Random (MAR). It is not MCAR because the rates are much different from each other. 

For the analysis of the hypothesis, it would be appropitate for H1 to drop the `Time` missing values, whereas for H2 and H3 it would be appropiate to drop the missing values of `Before` and `Logic`???? **I DON'T REMEMBER!**

**Maybe we'll need to change this:**

For the new dataset where we need to delete both missing values from `Before`and `Logic` we will need to generalize the dataset to class 4 to 8, and not the full sample. 

Furthermore, for the new dataset where the missing values from `Time` are deleted, we will need to consider that the results could be only generalised to the ones who completed the puzzle, whereas no claims can be made about non-completers.

Case deletion of missing values from `Before` and `Logic`

In [17]:
df = df.drop('Missing', axis=1)
new_df1 = df[(df["Logic"].notna()) & (df["Before"].notna())]
new_df1

,Before,Type,Logic,Class,Time,Correct
588,Within_3,Letters,Yes,4,75.0,Yes
589,No,Symbols,Indifferent,4,350.0,No
590,Within_3,Numbers,Yes,4,56.0,Yes
591,Within_3,Greek,Yes,4,211.0,Yes
592,No,Symbols,No,4,278.0,Yes
...,...,...,...,...,...,...
1693,Within_3,Letters,Indifferent,8,210.0,Yes
1694,Out_3,Letters,Yes,8,206.0,No
1695,Out_3,Numbers,Yes,8,163.0,Yes
1696,Out_3,Numbers,Yes,8,140.0,Yes


Case deletion of missing values from `Time`

In [18]:
new_df2 = df[df["Time"].notna()]
new_df2

,Before,Type,Logic,Class,Time,Correct
2,NaN,Greek,NaN,1,252.0,No
4,NaN,Letters,NaN,1,88.0,No
7,NaN,Letters,NaN,1,160.0,Yes
8,NaN,Symbols,NaN,1,405.0,Yes
10,NaN,Symbols,NaN,1,361.0,Yes
...,...,...,...,...,...,...
1693,Within_3,Letters,Indifferent,8,210.0,Yes
1694,Out_3,Letters,Yes,8,206.0,No
1695,Out_3,Numbers,Yes,8,163.0,Yes
1696,Out_3,Numbers,Yes,8,140.0,Yes


Case deletion of each and every missing value

In [19]:
new_df3 = df.dropna(axis= 0)
new_df3

,Before,Type,Logic,Class,Time,Correct
588,Within_3,Letters,Yes,4,75.0,Yes
589,No,Symbols,Indifferent,4,350.0,No
590,Within_3,Numbers,Yes,4,56.0,Yes
591,Within_3,Greek,Yes,4,211.0,Yes
592,No,Symbols,No,4,278.0,Yes
...,...,...,...,...,...,...
1693,Within_3,Letters,Indifferent,8,210.0,Yes
1694,Out_3,Letters,Yes,8,206.0,No
1695,Out_3,Numbers,Yes,8,163.0,Yes
1696,Out_3,Numbers,Yes,8,140.0,Yes


In [20]:
new_df1.to_csv('3SudokuCombined_deleted_before_logic.csv', index = False)

In [21]:
new_df2.to_csv('3SudokuCombined_deleted_time.csv', index = False)

In [22]:
new_df3.to_csv('3SudokuCombined_deleted_all.csv', index = False)